# Notebook 3: Multi-Pass Noise Recovery Pipeline
---
## Strategy
A **7-pass architecture** that progressively refines clusters:
- Pass 0: Initial strict clustering
- Pass 1: Deduplication (merge syntactic variants)
- Pass 2: Noise recovery via cosine similarity
- Pass 3-5: Hierarchical structure building
- Pass 6: Granularity optimization (reduce to ~35 topics)
- Pass 7: Label refinement

## Flow


## When to Use
- When you need MAXIMUM topic coverage (low noise)
- When you want a hierarchy tree for nested navigation
- Best for dashboard/UI-driven search interfaces

## Key Insight
- Uses  to force noise docs
  into clusters only if similarity > 65%
- Produces a  showing parent-child relationships


In [ ]:
!pip install -q bertopic sentence-transformers hdbscan umap-learn pandas scikit-learn
print("Ready")

In [ ]:
import pandas as pd, numpy as np, re, html, json
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import umap, hdbscan
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

In [ ]:
DOMAIN_STOPS = {"abp","shorts","viral","subscribe","youtube","news","live","at2","n18v"}
ENTITY_MAP = [(r"\bpm\s+modi\b|\bmodi\s+ji\b|\bnarendra\s+modi\b", " pm_modi "),
              (r"\brahul\s+gandhi\b", " rahul_gandhi "),
              (r"\bdonald\s+trump\b", " donald_trump ")]

def clean(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    tokens = [t for t in text.lower().split() if not (t.isascii() and len(t)<=2) and t not in DOMAIN_STOPS]
    return " ".join(tokens).strip()

def ner_clean(text):
    for pat, repl in ENTITY_MAP:
        text = re.sub(pat, repl, text, flags=re.I)
    return clean(text)

# --- UNIVERSAL DATA LOADER ---
import pandas as pd
import os

# Get the file extension
file_ext = os.path.splitext(filename)[1].lower()

if file_ext == '.xlsx':
    print("📂 Detected Excel file. Using read_excel...")
    df = pd.read_excel(filename)
elif file_ext == '.csv' or file_ext == '.txt':
    print("📂 Detected Text/CSV file. Using read_csv with encoding recovery...")
    try:
        df = pd.read_csv(filename, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(filename, encoding='latin1')
        except:
            df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')
else:
    print(f"⚠️ Unknown format {file_ext}. Attempting generic read...")
    df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')

# Ensure we have a title column
if "title" not in df.columns:
    if len(df.columns) == 1:
        df.columns = ["title"]
    else:
        df.columns = ["title"] + list(df.columns[1:])
# ------------------------------
if "title" not in df.columns: df.columns=["title"]+list(df.columns[1:])
df["ner_text"] = df["title"].fillna("").astype(str).map(ner_clean)
df = df[df["ner_text"]!=""].reset_index(drop=True)

embedder = SentenceTransformer("sentence-transformers/LaBSE")
docs = df["ner_text"].tolist()
embeddings = embedder.encode(docs, show_progress_bar=True, batch_size=32)
print(f"Embedded {len(df)} docs")

## Pass 0: Initial Strict Clustering

In [ ]:
model = BERTopic(embedding_model=embedder, min_topic_size=15, nr_topics="auto", verbose=True)
topics, probs = model.fit_transform(docs, embeddings)
print(f"Pass 0: {len(set(topics))-1} topics, {round(100*(np.array(topics)==-1).mean(),1)}% noise")

## Pass 1 & 6: Deduplication + Granularity Optimization
Merge similar topics down to ~35 main clusters.

In [ ]:
# Force merge to target granularity
model.reduce_topics(docs, nr_topics=35)
topics = model.topics_
print(f"Pass 1+6: {len(set(topics))-1} topics, {round(100*(np.array(topics)==-1).mean(),1)}% noise")

## Pass 2: Noise Recovery
Force-assign noise to nearest cluster if cosine similarity > 0.65.

In [ ]:
new_topics = model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0.65)
model.update_topics(docs, topics=new_topics)
topics = new_topics
print(f"Pass 2: {len(set(topics))-1} topics, {round(100*(np.array(topics)==-1).mean(),1)}% noise")

## Pass 3-5: Hierarchical Structure

In [ ]:
hier = model.hierarchical_topics(docs)
tree = model.get_topic_tree(hier)
print(tree[:2000])  # Print first 2000 chars of hierarchy

## Final Export

In [ ]:
## FINAL CLEAN STANDARDIZED EXPORT
# 1. Create a CLEAN Topic Label Map (No numbers, Space-separated, Title-case)
topic_info = model.get_topic_info()
label_map = {}
for _, row in topic_info.iterrows():
    t_id = row['Topic']
    raw_name = row['Name']
    # Remove the "0_" or "-1_" prefix and clean up underscores
    if "_" in raw_name:
        clean_name = raw_name.split("_", 1)[-1].replace("_", " ").title()
    else:
        clean_name = raw_name.title()
    label_map[t_id] = clean_name

df['topic_label'] = df['topic_id'].map(label_map)

# 2. Reorganize Columns
if 'ner_text' not in df.columns: df['ner_text'] = df['clean_text']
final_df = df.rename(columns={'title': 'raw_text'})

# 3. Selection (Exactly 5 columns)
cols_to_keep = ['raw_text', 'clean_text', 'ner_text', 'topic_id', 'topic_label']
final_df = final_df[cols_to_keep]

# 4. Save and Download
output_fn = "final_clustering_results.csv"
final_df.to_csv(output_fn, index=False)
print(f"Success! Exported {len(final_df)} titles with clean, unique labels.")
from google.colab import files
files.download(output_fn)